# 1. EDA and data cleaning

In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [4]:
df = pd.read_csv("/content/patents_g06n3_wide.csv")
df.head()

df.info()

In [14]:
duplicate_text = df[df.duplicated(
    subset=["title", "abstract"],
    keep=False
)]

print(f"Duplicate patents by text: {len(duplicate_text)}")

duplicate_text[[
    "publication_number",
    "title"
]].sort_values("title")

Duplicate patents by text: 83


,publication_number,title
2297,US-10650099-B2,Architecture and processes for computer learni...
1654,US-10614165-B2,Architecture and processes for computer learni...
1103,US-2024211747-A1,Automated ground truth generation using a neur...
1500,US-12524670-B2,Automated ground truth generation using a neur...
1718,US-2024154998-A1,Automated learning and detection of web bot tr...
...,...,...
2347,US-2025245297-A1,Systems and methods of sensor data fusion
1296,US-2025144805-A1,Systems and methods of sensor data fusion
1586,US-12541574-B2,Systems and methods of sensor data fusion
797,US-2019318225-A1,Tuning of loop orders in blocked dense basic l...


In [15]:
print(df.duplicated(subset="publication_number").sum())

print(df.duplicated(subset="title").sum())

print(df.duplicated(subset=["title", "abstract"]).sum())

print(df.duplicated(subset=["title", "abstract", "publication_number"]).sum())

0
71
45
0
718


In [5]:
def clean_text(text):
    text = text.lower() #lowercase
    text = re.sub(r"\n", " ", text) #remove new lines
    text = re.sub(r"[^a-z0-9\s]", " ", text) #remove special characters
    text = re.sub(r"\s+", " ", text) #remove extra spaces
    return text.strip()


# 2. THE ENGINE

In [12]:
class PatentSimilarityEngine:
    def __init__(self, max_features: int = 20000, ngram_range=(1, 2)):
        """
        max_features: caps vocabulary size (keeps matrix manageable on large corpora)
        ngram_range=(1,2): captures both single words and two-word phrases
                            (e.g. "neural network", "machine learning"),
                            which matters a lot for patent text.
        """
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            stop_words="english",
            min_df=2,        # ignore terms that appear in only 1 document (noise)
            max_df=0.95,     # ignore terms that appear in >85% of docs (too generic)
        )
        self.tfidf_matrix = None
        self.df = None


    def fit(self, df: pd.DataFrame, id_col: str, text_cols: list):
        """
        df: your patents dataframe
        id_col: column that uniquely identifies a patent (e.g. 'publication_number')
        text_cols: columns to combine into the text used for similarity
                   (e.g. ['title', 'abstract'])
        """
        self.df = df.reset_index(drop=True).copy()
        self.id_col = id_col

        # Combine chosen text columns into one field per patent
        combined = self.df[text_cols].fillna("").agg(" ".join, axis=1)
        self.df["_clean_text"] = combined.apply(clean_text)

        # Build the TF-IDF matrix: one row per patent, one column per term
        self.tfidf_matrix = self.vectorizer.fit_transform(self.df["_clean_text"])
        print(f"Fitted TF-IDF on {self.tfidf_matrix.shape[0]} patents, "
              f"vocabulary size = {self.tfidf_matrix.shape[1]}")
        return self


    def find_similar_to_existing(self, patent_id: str, top_n: int = 5):
        """
        Find the top_n most similar patents to a patent ALREADY in the dataset,
        identified by its id (e.g. publication_number).
        """
        matches = self.df.index[self.df[self.id_col] == patent_id].tolist()
        if not matches:
            raise ValueError(f"No patent found with id '{patent_id}'")
        idx = matches[0]

        query_vector = self.tfidf_matrix[idx]
        sims = cosine_similarity(query_vector, self.tfidf_matrix).flatten()

        return self._top_results(sims, exclude_idx=idx, top_n=top_n)


    def find_similar_to_new_text(self, title: str, abstract: str, top_n: int = 5):
        """
        Find the top_n most similar patents to a BRAND NEW patent description
        that is not yet in the dataset (e.g. a draft application you're
        checking for novelty/overlap).
        """
        combined = clean_text(f"{title} {abstract}")
        query_vector = self.vectorizer.transform([combined])
        sims = cosine_similarity(query_vector, self.tfidf_matrix).flatten()

        return self._top_results(sims, exclude_idx=None, top_n=top_n)


    def _top_results(self, sims: np.ndarray, exclude_idx, top_n: int):
        order = np.argsort(sims)[::-1]  # descending similarity
        results = []
        for idx in order:
            if exclude_idx is not None and idx == exclude_idx:
                continue
            results.append({
                self.id_col: self.df.loc[idx, self.id_col],
                "title": self.df.loc[idx, "title"],
                "similarity_score": round(float(sims[idx]), 4),
            })
            if len(results) >= top_n:
                break
        return pd.DataFrame(results)


    def save(self, path: str):
        """Persist the fitted engine (vectorizer + matrix + df) to disk."""
        with open(path, "wb") as f:
            pickle.dump({
                "vectorizer": self.vectorizer,
                "tfidf_matrix": self.tfidf_matrix,
                "df": self.df,
                "id_col": self.id_col,
            }, f)

    @classmethod
    def load(cls, path: str):
        """Load a previously-saved engine so you don't have to refit every run."""
        with open(path, "rb") as f:
            data = pickle.load(f)
        engine = cls()
        engine.vectorizer = data["vectorizer"]
        engine.tfidf_matrix = data["tfidf_matrix"]
        engine.df = data["df"]
        engine.id_col = data["id_col"]
        return engine

In [7]:

# 3. NOVELTY / "SIMILARITY ALERT" HELPER
def check_novelty(engine: PatentSimilarityEngine, title: str, abstract: str,
                   threshold: float = 0.35, top_n: int = 5):
    """
    Convenience wrapper for the actual use case: "does this new patent
    idea overlap with existing ones?" Prints a clear verdict plus the
    supporting evidence (closest matches).
    """
    results = engine.find_similar_to_new_text(title, abstract, top_n=top_n)
    max_score = results["similarity_score"].max() if len(results) else 0.0

    print(f"\n{'='*70}")
    print(f"NOVELTY CHECK: {title}")
    print(f"{'='*70}")
    if max_score >= threshold:
        print(f"POTENTIAL OVERLAP DETECTED (top similarity = {max_score:.3f}, "
              f"threshold = {threshold})")
    else:
        print(f" No strong overlap found (top similarity = {max_score:.3f}, "
              f"threshold = {threshold})")
    print("\nClosest existing patents:")
    print(results.to_string(index=False))
    print(f"{'='*70}\n")
    return results

# 4. DEMO / ENTRY POINT

In [13]:
# Fit the engine on title + abstract ---
engine = PatentSimilarityEngine(max_features=20000, ngram_range=(1, 2))
engine.fit(df, id_col="publication_number", text_cols=["title", "abstract"])

# Example 1: find patents similar to one already in the dataset ---
sample_id = df["publication_number"].iloc[0]
print(f"\nTop matches for existing patent {sample_id}:")
print(engine.find_similar_to_existing(sample_id, top_n=5).to_string(index=False))

# Example 2: check a brand-new patent idea against the corpus ---
new_title = "Neural network training using synthetic sensor data for autonomous vehicles"
new_abstract = (
    "A system trains a neural network model using a combination of real "
    "sensor data collected from vehicles and synthetically generated data "
    "from a driving simulator, improving generalization for autonomous "
    "vehicle perception tasks."
)
check_novelty(engine, new_title, new_abstract, threshold=0.3, top_n=5)

# Save the fitted engine so you don't have to re-run TF-IDF every time ---
engine.save("/content/models/patent_engine.pkl")
print("Engine saved to patent_engine.pkl")

Fitted TF-IDF on 3000 patents, vocabulary size = 20000

Top matches for existing patent US-2024119857-A1:
publication_number                                                                                              title  similarity_score
  US-2024135197-A1                                                      Generating enriched scenes using scene graphs            0.3391
  US-2022314993-A1                                                                      Top-down scene discrimination            0.3094
    US-11288884-B2                                  UAV real-time path planning method for urban scene reconstruction            0.3039
  US-2024221312-A1 Systems and methods for generating and/or using 3-dimensional information with one or more cameras            0.2661
  US-2023041501-A1                                    Policy neural network training using a privileged expert policy            0.2318

NOVELTY CHECK: Neural network training using synthetic sensor data for autono